In [24]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import cross_val_score

df = pd.read_csv('../winequality-red-imputed.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   float64
dtypes: float64(12)
memory usage: 150.0 KB


In [25]:
X = df.drop(columns=['quality'])
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



### First, I'll use all features with the 'Support Vector Machine' (RBF and Linear kernels), applying both 'One-vs-Rest' and 'One-vs-One' strategies. I will also evaluate the variance and bias to assess the base model's performance initially.

In [26]:
# Function to print bias and variance indicators
def check_bias_variance(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

    accuracy_diff = abs(train_accuracy - test_accuracy)

    # Checking bias-variance balance
    if accuracy_diff > 0.05:
        print("Potential high variance (overfitting)\n")
    elif accuracy_diff < -0.05: 
        print("Potential high bias (underfitting)\n")
    elif accuracy_diff <= 0.05:  
        if train_accuracy < 0.7:
            print("Model is balanced but has low overall accuracy (potential underfitting)\n")
        else:
            print("Model is balanced\n")


In [27]:
# Linear
ovr_model_linear = OneVsRestClassifier(SVC(kernel='linear'))

ovr_model_linear.fit(X_train, y_train)

y_train_pred = ovr_model_linear.predict(X_train)
y_test_pred = ovr_model_linear.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsRestClassifier LINEAR:')
check_bias_variance(train_accuracy, test_accuracy)

print("=" * 30)
#RBF
ovr_model_rbf = OneVsRestClassifier(SVC(kernel='rbf'))

ovr_model_rbf.fit(X_train, y_train)

y_train_pred = ovr_model_rbf.predict(X_train)
y_test_pred = ovr_model_rbf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsRestClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)

OneVsRestClassifier LINEAR:
Training Accuracy: 55.043%
Testing Accuracy: 53.125%
Model is balanced but has low overall accuracy (potential underfitting)

OneVsRestClassifier RBF:
Training Accuracy: 53.87%
Testing Accuracy: 49.688%
Model is balanced but has low overall accuracy (potential underfitting)



In [28]:
from sklearn.multiclass import OneVsOneClassifier

# Linear
ovo_model_linear = OneVsOneClassifier(SVC(kernel='linear'))

ovo_model_linear.fit(X_train, y_train)

y_train_pred = ovo_model_linear.predict(X_train)
y_test_pred = ovo_model_linear.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsOneClassifier LINEAR : ')
check_bias_variance(train_accuracy, test_accuracy)
print("=" * 30)
# RBF
ovo_model_rbf = OneVsOneClassifier(SVC(kernel='rbf'))

ovo_model_rbf.fit(X_train, y_train)

y_train_pred = ovo_model_rbf.predict(X_train)
y_test_pred = ovo_model_rbf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsOneClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)


OneVsOneClassifier LINEAR : 
Training Accuracy: 59.578%
Testing Accuracy: 56.875%
Model is balanced but has low overall accuracy (potential underfitting)

OneVsOneClassifier RBF:
Training Accuracy: 51.056%
Testing Accuracy: 50.625%
Model is balanced but has low overall accuracy (potential underfitting)



### Second : As I've noticed all models are balanced but the accuracy is lower than the target 70%, so I'll go with the selected Features from the data preprocessing to see if it's going to increase or not also I'll use cross-Validation to make sure the performance holds across different train-test splits and the model still balanced  

In [29]:
'''
Selected Features classification:
 Index(['volatile acidity', 'total sulfur dioxide', 'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'total sulfur dioxide', 'alcohol']
X_selected = df[selected_features]
y = df['quality']
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

### Third : I'll do the first step again but with the selected features...

In [30]:
# Linear
ovr_model_linear = OneVsRestClassifier(SVC(kernel='linear'))

ovr_model_linear.fit(X_train, y_train)

y_train_pred = ovr_model_linear.predict(X_train)

test_accuracy = cross_val_score(OneVsRestClassifier(SVC(kernel='linear')), X_selected, y, cv=5).mean()
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f'OneVsRestClassifier LINEAR:')
check_bias_variance(train_accuracy, test_accuracy)

OneVsRestClassifier LINEAR:
Training Accuracy: 53.792%
Testing Accuracy: 53.407%
Model is balanced but has low overall accuracy (potential underfitting)



In [31]:
#RBF
ovr_model_rbf = OneVsRestClassifier(SVC(kernel='rbf'))

ovr_model_rbf.fit(X_train, y_train)

y_train_pred = ovr_model_rbf.predict(X_train)

test_accuracy = cross_val_score(OneVsRestClassifier(SVC(kernel='rbf')), X_selected, y, cv=5).mean()
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsRestClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)

OneVsRestClassifier RBF:
Training Accuracy: 52.854%
Testing Accuracy: 52.91%
Model is balanced but has low overall accuracy (potential underfitting)



In [32]:
# Linear
ovo_model_linear = OneVsOneClassifier(SVC(kernel='linear'))

ovo_model_linear.fit(X_train, y_train)

y_train_pred = ovo_model_linear.predict(X_train)

test_accuracy = cross_val_score(OneVsOneClassifier(SVC(kernel='linear')), X_selected, y, cv=5).mean()
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsOneClassifier LINEAR : ')
check_bias_variance(train_accuracy, test_accuracy)

OneVsOneClassifier LINEAR : 
Training Accuracy: 58.092%
Testing Accuracy: 57.038%
Model is balanced but has low overall accuracy (potential underfitting)



In [33]:
# RBF
ovo_model_rbf = OneVsOneClassifier(SVC(kernel='rbf'))

ovo_model_rbf.fit(X_train, y_train)

y_train_pred = ovo_model_rbf.predict(X_train)

test_accuracy = cross_val_score(OneVsOneClassifier(SVC(kernel='rbf')), X_selected, y, cv=5).mean()
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f'OneVsOneClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)


OneVsOneClassifier RBF:
Training Accuracy: 49.414%
Testing Accuracy: 49.029%
Model is balanced but has low overall accuracy (potential underfitting)

